# Lab 2: Local Features, Matching, Geometric Estimation & Template Matching

In Lab 1 we learned how an image lives in memory: as an `ndarray` with a shape, a dtype and a channel order, and how pixel-wise operations (thresholding, morphology) transform it. In this lab we move from *pixels* to *structure*: we will extract **distinctive points** from images, describe their local neighbourhood as **vectors**, **match** these vectors between images, and use the matches to estimate a **geometric transformation** (a homography). Finally, we look at the classical alternative that skips the "describe" step entirely: **template matching**.

The goal is, again, to understand *why* these methods work, *what* data structures flow between the steps.

**Lab 2 outline**
1. Template matching (correlation, normalized correlation, FFT, multi-scale, pyramidal coarse-to-fine)
2. Keypoint detection and descriptors (SIFT)
3. Keypoint matching (brute force, distance metrics, ratio test)
4. Homography estimation (DLT, RANSAC) and image warping
5. Synthesis: choosing the right tool

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# run this code to import the three primary libraries 
# (make sure a green checkmark appears, this should not take more than a few seconds)
import cv2
import numpy as np
import matplotlib.pyplot as plt

## Section 1: Template Matching

Keypoint-based matching needs texture and gives a *transformation*. If instead we know **exactly what the object looks like** (a fixed pattern at a known scale and orientation: a logo, a component on a PCB, a letter, a coin, an icon in a screenshot), a much simpler idea works: slide the template over the image and measure *how well it fits at every position*. No detector, no descriptor, no RANSAC.

Let the image be $I\in\mathbb{R}^{H\times W}$ (note that that this perfectly generalizes to a case where we have a 3D image) and the template $T\in\mathbb{R}^{h\times w}$. Template matching produces a **response map** $R$ where $R(x,y)$ measures the similarity between $T$ and the patch of $I$ whose **top-left corner** is at $(x,y)$. Only positions where the template fully fits inside the image are evaluated ("valid" mode), hence

$$R\in\mathbb{R}^{(H-h+1)\times(W-w+1)}.$$

(**Pitfall:** the location of the best match is the *top-left* corner of the patch, not its centre; add $(w/2, h/2)$ for the centre.)

**Similarity measures.** Define the patch $I_{x,y}(i,j)=I(x+j,\,y+i)$.

The most useful one is the **zero-normalized cross-correlation (ZNCC)**, `TM_CCOEFF_NORMED`:

$$R(x,y)=\frac{\sum_{i,j}\big(T(i,j)-\bar T\big)\big(I_{x,y}(i,j)-\bar I_{x,y}\big)}{\sqrt{\sum_{i,j}\big(T(i,j)-\bar T\big)^2\;\sum_{i,j}\big(I_{x,y}(i,j)-\bar I_{x,y}\big)^2}}\in[-1,1]$$

It is the **cosine similarity / Pearson correlation** between the flattened template and the flattened patch. This gives it an important property: it is invariant to **affine intensity changes** $I\mapsto aI+b$ (global contrast and brightness), so it has a clear geometric meaning (1 = identical up to gain and offset, 0 = uncorrelated, −1 = inverted contrast), unlike `TM_CCORR`, which is biased towards bright regions (a white patch has a larger inner product with *any* template than a perfect but darker match).

#### The OpenCV functions and data structures
- **`R = cv2.matchTemplate(image, templ, method, mask=None)`**: `image` and `templ` must have the same dtype (`uint8` or `float32`) and number of channels, `templ` must be at most as large as `image`. Returns `float32` of shape `(H-h+1, W-w+1)`. As mask you can optionally pass a binary image that is 1 for the foreground and 0 for the background of the template, indicating the region of interest in the template.
- **`min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(R)`**: value and **`(x0, y0)`** location of the global minimum and maximum. Use `max_loc` as we want the best match.
- Cutting out a template with slicing `best_match = img[y0:y1, x0:x1]` gives a **view** (see previous lab session).

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (but run this code)

from typing import Tuple

# these are the paths to the template images of the four characters
template_paths: Tuple[str] = (
    'wanted_assets/Luigi_template.png', 
    'wanted_assets/Wario_template.png', 
    'wanted_assets/Yoshi_template.png', 
    "wanted_assets/Mario_template.png",
    )

# this is the color of the background in the template images (yellow-ish), we can ignore these pixels when we want to find the best match
BG_YELLOW = (255, 231, 66)

# these are the paths to the target images where we want to find the character (the name at the end of the file path)
target_paths: Tuple[str] = (
    'wanted_assets/level_1_find_Yoshi.png',
    'wanted_assets/level_2_find_Mario.png',
    'wanted_assets/level_3_find_Mario.png',
    'wanted_assets/level_4_find_Yoshi.png',
    'wanted_assets/level_5_find_Luigi.png',
    'wanted_assets/level_6_find_Yoshi.png',
    'wanted_assets/level_7_find_Luigi.png',
    'wanted_assets/level_8_find_Wario.png',
    'wanted_assets/level_9_find_Luigi.png',
    'wanted_assets/level_10_find_Mario.png',
    'wanted_assets/level_11_find_Wario.png',
    'wanted_assets/level_12_find_Luigi.png',
    'wanted_assets/level_13_find_Luigi.png',
    'wanted_assets/level_14_find_Luigi.png',
    'wanted_assets/level_15_find_Luigi.png',
)

In [ ]:
"""WRITE YOUR CODE HERE"""

# implement the template matching function here and return the best matching coordinates!

def localize_character(template_image: np.ndarray, target_image: np.ndarray, template_fg_mask: np.ndarray) -> tuple[int, int]:
    """
    Find the best match for the template image (while using the foreground mask) in the target image and return the top-left corner coordinates.
    
    Args:
        template_image: RGB image of the character template.
        target_image: RGB image of the game level.
        template_fg_mask: Binary mask (255 for foreground, 0 for background).
        
    Returns:
        tuple[int, int]: The top-left corner coordinates (x, y) of the best match.
        also return the response image R (the result of cv2.matchTemplate) for visualization.
    """

    raise NotImplementedError("Your code goes here!")

    
    return (0,0), R

We can look at the response map in two ways:

- **2D:** as a regular image, where the strongest responses are the brightest pixels.
- **3D:** as a surface plot, where x and y are the pixel position and the height is the response. The higher the response, the higher the peak.

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
def plot_response_3d(R, cmap="viridis", elev=45, azim=-60):
    plt.imshow(R, cmap='hot')
    plt.show()
    R = np.asarray(R, dtype=float)
    y, x = np.mgrid[0:R.shape[0], 0:R.shape[1]]   # pixel coordinates

    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(projection="3d")
    surf = ax.plot_surface(x, y, R, cmap=cmap, linewidth=0)
    ax.set_xlabel("x (pixel)")
    ax.set_ylabel("y (pixel)")
    ax.set_zlabel("response")
    ax.invert_yaxis()                 # row 0 at the top, like the image
    ax.view_init(elev=elev, azim=azim)
    fig.colorbar(surf, ax=ax, shrink=0.6, label="response")
    fig.show()
    plt.show()

Run the visualisation on one image:

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (but run this code)
target_path = target_paths[0]
# load the target image and convert it to RGB
target_image = cv2.cvtColor(cv2.imread(target_path), cv2.COLOR_BGR2RGB)

# extract the name of the character to find
find = target_path.split('_')[-1][:-4]

# construct the path to the template image 
template_path = f"wanted_assets/{find}_template.png"

# load the template image and convert it to RGB
template_image = cv2.cvtColor(cv2.imread(template_path), cv2.COLOR_BGR2RGB)

# create the foreground mask of the template image
template_fg_mask = (template_image != BG_YELLOW).any(axis=2).astype(np.uint8) * 255

# find the best match between the template and the target image 
# (return the top-left coordinates on the target image of the best match)
(x, y), R = localize_character(template_image, target_image, template_fg_mask)
plot_response_3d(R, cmap="hot", elev=45, azim=-60)


In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (but run this code)

# implement the function localize_character above and run this cell to see the results
# does the detection always work, when does it fail? how can we improve it?

from matplotlib import pyplot as plt

for target_path in target_paths:
    # load the target image and convert it to RGB
    target_image = cv2.cvtColor(cv2.imread(target_path), cv2.COLOR_BGR2RGB)
    
    # extract the name of the character to find
    find = target_path.split('_')[-1][:-4]
    
    # construct the path to the template image 
    template_path = f"wanted_assets/{find}_template.png"

    # load the template image and convert it to RGB
    template_image = cv2.cvtColor(cv2.imread(template_path), cv2.COLOR_BGR2RGB)

    # create the foreground mask of the template image
    template_fg_mask = (template_image != BG_YELLOW).any(axis=2).astype(np.uint8) * 255

    # find the best match between the template and the target image 
    # (return the top-left coordinates on the target image of the best match)
    (x, y), R = localize_character(template_image, target_image, template_fg_mask)

    # draw the best match template on the target image (from top-left to bottom-right)
    drawing = target_image.copy()
    th, tw = template_fg_mask.shape[:2]
    cv2.rectangle(drawing, (x, y), (x + tw, y + th), (255, 0, 0), 2)
    plt.imshow(drawing)
    plt.title(f' detection result for finding "{find}"')
    plt.axis("off")
    plt.show()


## Section 2: Keypoint Detection & Descriptors

### 2.1 What is a good keypoint? Corners and the structure tensor

A **keypoint** (interest point, feature) is a location in the image that can be found again, reliably, in another image of the same scene. A good keypoint detector should be:

- **Repeatable:** the same physical point is detected under changes in viewpoint, scale, rotation and illumination.
- **Distinctive / localizable:** the neighbourhood of the point must be *unique*, so that it can be located precisely.

Consider a small window $W$ around a pixel and shift it by $(u, v)$. The change in intensity is

$$E(u,v) = \sum_{(x,y)\in W} w(x,y)\,\big[I(x+u,\,y+v) - I(x,y)\big]^2 \;\approx\; \begin{bmatrix}u & v\end{bmatrix} M \begin{bmatrix}u \\ v\end{bmatrix}$$

We look at the local structure (gradients) of the intensity around the point $(x,y)$ 

**Dataflow:** image $\rightarrow$ gradients $\rightarrow$ products $I_x^2, I_y^2, I_xI_y$ $\rightarrow$ window-sum (blur) $\rightarrow$ response map $R$ of shape $(H,W)$ $\rightarrow$ threshold $\rightarrow$ **non-maximum suppression** (keep only local maxima) $\rightarrow$ list of coordinates. Note that the *detector output is first a dense map*, and only after thresholding + NMS does it become a *sparse list of points*.

Note that when we work with the structure in the image, we are not interested in the color but in the spatial gradient. The gradient is the directional derivative of the grayscale image intensity. Therefore, **we always convert the image to grayscale before detecting and describing keypoints**! 

### 2.2 Scale invariance and the SIFT detector

The formulation above is **rotation invariant** (the eigenvalues of $M$ do not change when the image is rotated), but *not* **scale invariant**: a corner at one scale looks like a smooth curve at another. The solution is to search for keypoints in **scale space**. By looking at the image at different scales (blurring with a Gaussian) we can find keypoints that are invariant to scale. 


### 2.3 From keypoint to descriptor

Detection tells us *where* to look; the **descriptor** encodes *what is there*: a fixed-length vector computed from the neighbourhood of the keypoint (scaled by `size` and rotated by `angle`, hence invariant to both). Two keypoints in different images that depict the same physical point should end up with **nearby vectors**. Matching therefore becomes a nearest-neighbour search in a vector space.

**SIFT descriptor (float, 128-D).** Take a $16\times16$ patch (scaled/rotated), split it into a $4\times4$ grid of cells. In each cell build an **8-bin orientation histogram** of gradients (magnitude-weighted, Gaussian-weighted from the centre, with trilinear interpolation). That gives $4\times4\times8 = 128$ numbers.



In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (define the file paths for the images, make sure these files are in the same directory as this notebook!)
# run this cell to set the paths
TEMPLATE_PATH = "assets/fitness_template.png"
TARGET_PATH = "assets/fitness_target.png"

# leave this commented for now, later you can uncomment them and re-apply the entire workflow on these images

# TEMPLATE_PATH = "assets/kinderbueno_image.png"
# TARGET_PATH = "assets/kinderbueno_target.png"

In [ ]:
"""WRITE YOUR CODE HERE"""

# load images and convert to rgb (these can be used for visualization with matplotlib)

# convert to grayscale (these can be used for keypoint detection)


In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (but run this code)
det_des: cv2.Feature2D = cv2.SIFT_create()

# this object can be used to detect keypoints in an image and compute its descriptors:
#kp, des = det_des.detectAndCompute(image=gray_img, mask=None)

In [ ]:
"""WRITE YOUR CODE HERE"""

# apply the SIFT object to the template and target images and get the keypoints and descriptors

# print the number of keypoints detected in both

# print the dimensions of the descriptors

# draw the keypoints on the template and target images

# show the images with matplotlib


## Section 3: Keypoint Matching

### 3.1 Matching as nearest-neighbour search

We now have two sets of descriptors: `des1` of shape `(N1, D)` (from the **query** image) and `des2` of shape `(N2, D)` (from the **train** image). For every row (keypoint description) of `des1` we look for the row of `des2` that is closest in descriptor space.

**Brute force matching** simply computes the full **distance matrix** $\mathbf{D}\in\mathbb{R}^{N_1\times N_2}$ with $D_{ij} = d(\mathbf{a}_i,\mathbf{b}_j)$ and takes the row-wise `argmin`. The cost is $O(N_1 N_2 D)$.

(there are alternatives to brute force matching, e.g. using a **kd-tree** or **flann**, these are faster, which might be usefull for real-time applications, but come at the cost of not being as accurate, as they do not find the global optimum)

**Distance metrics** (the metric must fit the descriptor type!):

- **$L_2$** (`cv2.NORM_L2`), for SIFT: $d(\mathbf{a},\mathbf{b}) = \lVert\mathbf{a}-\mathbf{b}\rVert_2$. 
- **Hamming** (`cv2.NORM_HAMMING`), for ORB/BRIEF/BRISK: the number of differing bits, $d(\mathbf{a},\mathbf{b}) = \operatorname{popcount}(\mathbf{a}\oplus\mathbf{b})$.

-> Using the wrong norm type (e.g. $L_2$ on packed binary descriptors) does not crash, it silently gives meaningless matches!

#### Why is the nearest neighbour not enough?
The nearest neighbour *always* exists, even if the true correspondence is not in the other image at all (occlusion, other part of the scene). We need criteria to **reject unreliable matches**:

**Lowe's ratio test:** for each query descriptor find the two nearest neighbours with distances $d_1 \le d_2$ and keep the match only if $\dfrac{d_1}{d_2} < \tau$ (typically $\tau\approx0.75$). A correct match has a clearly closer best neighbour than the runner-up; for an ambiguous (e.g. repeated texture) or non-existent match the two distances are similar and the ratio approaches 1. (In Lowe's original experiments, $\tau=0.8$ rejected ~90% of the false matches while losing only ~5% of the correct ones.)


In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (but run this code)

def lowe_ratio_test(matches: list[tuple[cv2.DMatch, cv2.DMatch]], ratio: float = 0.8) -> list[cv2.DMatch]:
    """Apply Lowe's ratio test to eliminate poor matches."""
    good_matches = []
    for matches_set in matches:
        best_match, second_best_match = matches_set[0], matches_set[1]
        if (best_match.distance / second_best_match.distance) < ratio:
            good_matches.append(best_match)
    return good_matches


In [19]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (but run this code)
def draw_the_good_matches(template, target, kp_template, kp_target, good_matches):
    # draw the good matches: template on the left, target on the right
    match_img = cv2.drawMatches(
        template, kp_template,          # left image + its keypoints
        target, kp_target,              # right image + its keypoints
        good_matches, None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,   # only show matched keypoints
    )

    plt.figure(figsize=(15, 8))
    plt.imshow(match_img)
    plt.title(f"{len(good_matches)} good matches")
    plt.axis("off")
    plt.show()

In [ ]:
"""WRITE YOUR CODE HERE"""

# create the bf matcher object (do not use cross_check)

# find the 2 nearest matches with knn

# print the total number of matches

# filter out ambiguous matches using the lowe_ratio_test function

# draw the good matches

# print the number of good matches


## Section 4: Homography Estimation & RANSAC

While we cleaned up the matches using Lowe's ratio test, we still have some outliers. To mathematically describe where exactly the template object is in the target image, we need to find a **homography** $\mathbf{H}$ that maps the template to the target. This is a $3\times3$ matrix $\mathbf{H}$. To estimate this matrix, we need to solve a system of linear equations, which is called the **Direct Linear Transform** (DLT). The DLT is the least squares solution, but this is prone to failure when there are outliers (which there are most of the time). To solve this, we use the **RANSAC** algorithm, this algorithm is also called **RANdom SAmple Consensus**, which is a *hypothesize-and-verify* loop that iteratively estimates the model and checks if it is robust enough. the function `cv2.findHomography` implements this loop, it returns both the matrix and a mask of inliers (the inliers are the matches that are considered correct).

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (but run this code)

from typing import Tuple

def ransac_info(template_keypoints: Tuple[cv2.KeyPoint], target_keypoints: Tuple[cv2.KeyPoint], matches: Tuple[cv2.DMatch]
    ) -> Tuple[np.ndarray, Tuple[cv2.DMatch]]:
    """No need to adapt this function!!!"""
    
    # extract coordinates of the matched keypoints
    tem = np.float32([template_keypoints[m.queryIdx].pt for m in matches])
    tar = np.float32([target_keypoints[m.trainIdx].pt for m in matches])

    # inlier_matches = tuple(match for match, inlier in zip(matches, inliers) if inlier)

    # find the homography matrix using RANSAC
    return cv2.findHomography(tem, tar, cv2.RANSAC)


def draw_homographized_template_outline(target_image: np.ndarray, template: np.ndarray, homography_matrix: np.ndarray,
    ) -> np.ndarray:
    """No need to adapt this function!!!"""
    
    # define the corner points of the template
    h,w = template.shape[:2]
    corner_points = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
   
    # warp the corner points using the homography matrix
    warped_corner_points = cv2.perspectiveTransform(corner_points,homography_matrix)
    
    return cv2.polylines(target_image,[np.int32(warped_corner_points)],True,(255,0,0),3, cv2.LINE_AA)


We can also show which matches RANSAC considered outliers when it calculated the homography.
`cv2.findHomography` returns a `mask` with a 1 for every inlier and a 0 for every outlier.
We use it to draw the inliers in green and the outliers in red:

In [ ]:
"""No need to adapt this function!!!"""
def show_ransac_results(template: np.ndarray, target: np.ndarray, kp_template: Tuple[cv2.KeyPoint], kp_target: Tuple[cv2.KeyPoint], good_matches: Tuple[cv2.DMatch], mask: np.ndarray):
    inliers = mask.ravel().tolist()
    outliers = [1 - m for m in inliers]
    flags = cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS

    img = cv2.drawMatches(template, kp_template, target, kp_target, good_matches, None,
                        matchColor=(255, 0, 0), matchesMask=outliers, flags=flags)
    img = cv2.drawMatches(template, kp_template, target, kp_target, good_matches, img,
                        matchColor=(0, 255, 0), matchesMask=inliers,
                        flags=flags | cv2.DrawMatchesFlags_DRAW_OVER_OUTIMG)

    plt.figure(figsize=(15, 8))
    plt.imshow(img)
    plt.title(f"RANSAC: {sum(inliers)} inliers (green), {sum(outliers)} outliers (red)")
    plt.axis("off")
    plt.show()

In [ ]:
"""WRITE YOUR CODE HERE"""

# use the good matches to fit the homography matrix using the ransac_info function

# draw the inliers and outliers using the show_ransac_results function

# draw the outline of the homography-transformed template on the target image using the draw_homographized_template_outline function


# plot the drawn image with matplotlib


Now that you completed the full keypoint detection pipeline, you should be able to apply it on any set of images.
Go back to the top of the notebook where you defined the image paths, uncomment the two lines that define paths of two other images, and run the code again.

the entire workflow should work very similar on this new set of images!

## Section 5: Synthesis

You now have two families of methods to find "the same thing" in another image. Use the cells below to compare them *on your own cases*.

| | Keypoints + descriptors + RANSAC (sections 1-3) | Template matching (section 4) |
|---|---|---|
| What is matched | local descriptors (sparse) | raw pixel patch at every position (dense) |
| Output | transformation $\mathbf{H}$ + inlier set | response map + location(s) |
| Invariances | scale, rotation, (moderate) viewpoint & illumination | only brightness/contrast (ZNCC); *not* scale/rotation/perspective, unless searched explicitly |
| Needs texture? | yes (needs corners/blobs) | works on low-texture objects as long as shape/intensity pattern is distinctive |
| Multiple instances | awkward (RANSAC finds the dominant model) | natural (threshold + NMS) |
| Cost | detection + $O(N_1N_2D)$ matching + RANSAC | $O(HWhw)$ per scale/angle; reduced by FFT/pyramids |
| Typical failure | repetitive patterns, blur, textureless objects, non-planar scenes (for $\mathbf{H}$) | scale/rotation change, occlusion, lighting *shape* change, clutter that resembles the template |

### OWN MINI PROJECT:

if you have time or want to practice, you can now try to implement one or both of the methods on a chosen case study. 
For example;
1. detecting a specific book in a bookstore
2. detecting a lego-box in a room
3. counting coins on a table
4. detecting pills in a medical image
and so on...